# Debt plan — budget + scenarios

Integrates **income**, **monthly budget**, **emergency fund**, and **multi-debt avalanche** payoff.

1. Copy `config/plan.example.yaml` → `config/plan.yaml` and enter your numbers.
2. Run all cells.

Compare scenarios side-by-side (balanced, lean lifestyle, sprint phases, safety-net-first).

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import load_plan, surplus, take_home_at, compare_scenarios, run_scenario

cfg_path = ROOT / "config" / "plan.yaml"
if not cfg_path.exists():
    cfg_path = ROOT / "config" / "plan.example.yaml"
    print("Using example plan — run ./scripts/init_local.sh for your own plan.yaml")

plan = load_plan(cfg_path)
inc = take_home_at(plan, 0)
b = plan["budget"]
debt_budget = surplus(inc, b["essentials"], b["lifestyle"], b["emergency_fund"])
total_debt = sum(d["balance"] for d in plan["debts"])

print(f"Take-home (month 1): ${inc:,.0f}/mo  |  Total debt: ${total_debt:,.0f}")
if plan.get("income", {}).get("phases"):
    later = take_home_at(plan, 1)
    if later != inc:
        print(f"Take-home (month 2+): ${later:,.0f}/mo  (income.phases active)")
print(f"Budget: essentials ${b['essentials']:,.0f} + lifestyle ${b['lifestyle']:,.0f} + EF ${b['emergency_fund']:,.0f}")
print(f"→ ${debt_budget:,.0f}/mo available for debt (default scenario, month 1)")
pd.DataFrame(plan["debts"])

## Compare all scenarios

In [ ]:
comparison = compare_scenarios(plan)
comparison

## Deep dive — pick a scenario

Change `SCENARIO` to any name from the table above.

In [ ]:
SCENARIO = plan["scenarios"][0]["name"]
scenario = next(s for s in plan["scenarios"] if s["name"] == SCENARIO)
result = run_scenario(plan, scenario)
hist = result["history"]

print(f"=== {SCENARIO} ===")
print(f"Debt-free in {result['debt_free_month']} months (~{result['summary']['years']} years)")
print(f"Total interest: ${result['summary']['total_interest']:,.2f}")
print(f"Emergency fund at end: ${result['ef_at_end']:,.2f}")
for name, month in result["summary"]["cleared_month"].items():
    print(f"  {name} cleared month {month}")
hist[["month", "debt_budget", "total_debt", "emergency_fund", "interest"]].head(12)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
hist.plot(x="month", y="total_debt", ax=axes[0], legend=False, color="#185")
axes[0].set_ylabel("Total debt $")
axes[0].set_title(f"{SCENARIO}: debt payoff")
axes[0].grid(alpha=0.3)

hist.plot(x="month", y="emergency_fund", ax=axes[1], legend=False, color="#2a8")
axes[1].axhline(plan["emergency_fund"]["target"], ls="--", color="#888", label="EF target")
axes[1].set_ylabel("Emergency fund $")
axes[1].set_xlabel("Month")
axes[1].legend()
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Overlay scenarios — total debt over time

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for scenario in plan["scenarios"]:
    r = run_scenario(plan, scenario)
    r["history"].plot(x="month", y="total_debt", ax=ax, label=scenario["name"])
ax.set_title("Total debt by scenario")
ax.set_xlabel("Month")
ax.set_ylabel("$")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()